# cerberus-neuro — v0 paired training run

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/02_train.ipynb)

Train two paired models on the same scoped subset and same train/val split:

1. **`CerberusModel`** — brightfield-only multi-task (cell type + line condition + virtual staining). Pushed to `patrickjreed/cerberus-neuro-v0`.
2. **`BaselineDiseaseClassifier`** — 6-channel single-task disease classifier. Pushed to `patrickjreed/cerberus-neuro-v0-baseline`.

The headline v0 metric is the gap between the two disease accuracies, reported in `notebooks/03_eval.ipynb`. This notebook runs the training; v0 step 6 evaluates.

Runtime expectations on Colab Free T4: ~60–90 min total per model at `n_epochs=25` (first epoch dominated by S3 → Drive cache fill at ~10–30 min; subsequent epochs ~1–2 min each on cached data). Both models combined: ~2–3 h.

## 1. Setup

In [ ]:
!pip install -q --upgrade "cerberus-neuro[training] @ git+https://github.com/PatrickJReed/cerberus-neuro.git@main" imagecodecs

import sys
for _m in list(sys.modules):
    if _m == 'cerberus_neuro' or _m.startswith('cerberus_neuro.'):
        del sys.modules[_m]
print('install + cache purge done')

In [ ]:
import os
from pathlib import Path
import torch

try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('HF login: OK (Colab secret)')
except Exception as e:
    print(f'HF login skipped ({type(e).__name__}); HF push will be disabled')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path('/content/drive/MyDrive/cerberus-neuro/cache/v0')
except Exception:
    CACHE_DIR = Path.home() / '.cache' / 'cerberus-neuro' / 'v0'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'cache dir: {CACHE_DIR}')
print(f'GPU: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

## 2. Manifest + train/val split

Build the full v0 manifest (4 batches, excludes 63x neurons), subsample to ~16 wells per cell type × 4 sites per well, then split wells stratified by `(cell_type, condition)`. The same split is used for both Cerberus and the baseline so the disease comparison is apples-to-apples.

In [ ]:
from cerberus_neuro.data import (
    build_manifest, subset_manifest, well_level_split, NeuroPaintingDataset,
)

manifest = build_manifest(CACHE_DIR)
print(f'full v0 manifest: {manifest.shape}')

subset = subset_manifest(manifest, wells_per_cell_type=48, sites_per_well=9, seed=0)
print(f'subset (balanced control/deletion): {subset.shape}')
print(subset.groupby(['Metadata_cell_type', 'Metadata_line_condition']).size().to_string())

train_manifest, val_manifest = well_level_split(subset, val_frac=0.2, seed=0)
print(f'\ntrain: {train_manifest.shape[0]} sites '
      f'({train_manifest[["Metadata_Plate", "Metadata_Well"]].drop_duplicates().shape[0]} wells)')
print(f'val:   {val_manifest.shape[0]} sites '
      f'({val_manifest[["Metadata_Plate", "Metadata_Well"]].drop_duplicates().shape[0]} wells)')

## 3. Train the Cerberus multi-task model

Brightfield input (1 channel) → cell-type logits + line-condition logits + 5-channel virtual fluorescence. Kendall uncertainty weighting balances the three losses; AdamW + cosine + AMP.

In [ ]:
from torch.utils.data import DataLoader

from cerberus_neuro.model import CerberusModel
from cerberus_neuro.training import TrainConfig, train

# TF32 matmul on Ampere/Ada GPUs (L4, A100). Free ~5-10% speedup.
torch.set_float32_matmul_precision('high')

BATCH_SIZE = 64           # bumped from 16 to better utilize L4 VRAM
CROPS_PER_SITE = 12
NUM_WORKERS = 4           # parallel data loading hides S3/decode latency
LR = 6e-4                 # sqrt-scaled from 3e-4 to compensate for 4x batch

train_ds = NeuroPaintingDataset(
    train_manifest, CACHE_DIR,
    crop_size=256, crops_per_site=CROPS_PER_SITE,
    min_cells_per_crop=1, augment=True,
)
val_ds = NeuroPaintingDataset(
    val_manifest, CACHE_DIR,
    crop_size=256, crops_per_site=CROPS_PER_SITE,
    min_cells_per_crop=1, augment=False, shuffle=False,
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0,
    pin_memory=torch.cuda.is_available(),
)

n_train_sites = len(train_manifest)
steps_per_epoch = max(1, (n_train_sites * CROPS_PER_SITE) // BATCH_SIZE)
print(f'steps_per_epoch={steps_per_epoch} (n_train_sites={n_train_sites}, crops_per_site={CROPS_PER_SITE}, batch={BATCH_SIZE})')

N_EPOCHS = 25
WARMUP_STEPS = max(1, int(0.05 * N_EPOCHS * steps_per_epoch))
print(f'n_epochs={N_EPOCHS}  warmup_steps={WARMUP_STEPS}  total_steps={N_EPOCHS * steps_per_epoch}')

cerberus_cfg = TrainConfig(
    n_epochs=N_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    lr=LR,
    weight_decay=1e-4,
    warmup_steps=WARMUP_STEPS,
    amp=torch.cuda.is_available(),
    log_every_steps=20,
    ckpt_every_steps=steps_per_epoch,  # one mid-epoch latest.pt write
    seed=0,
)

cerberus_dir = CACHE_DIR / 'cerberus'
cerberus_summary = train(
    model=CerberusModel(),
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cerberus_cfg,
    checkpoint_dir=cerberus_dir,
    hf_repo='patrickjreed/cerberus-neuro-v0',
    resume_from=cerberus_dir / 'latest.pt' if (cerberus_dir / 'latest.pt').exists() else None,
)
print('\ncerberus summary:', cerberus_summary)


## 4. Train the baseline disease classifier

Same encoder, same data split, same training recipe. Only differences: 6-channel input (BF + 5 fluorescence stacked), single line-condition head, no Kendall weighting (only one loss). This is the all-channel disease-accuracy upper bound the Cerberus model is compared against.

In [ ]:
from cerberus_neuro.model import BaselineDiseaseClassifier

baseline_cfg = TrainConfig(
    n_epochs=N_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    lr=LR,
    weight_decay=1e-4,
    warmup_steps=WARMUP_STEPS,
    amp=torch.cuda.is_available(),
    log_every_steps=20,
    ckpt_every_steps=steps_per_epoch,
    seed=0,
)

baseline_dir = CACHE_DIR / 'baseline'
baseline_summary = train(
    model=BaselineDiseaseClassifier(in_channels=6, n_classes=2),
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=baseline_cfg,
    checkpoint_dir=baseline_dir,
    hf_repo='patrickjreed/cerberus-neuro-v0-baseline',
    resume_from=baseline_dir / 'latest.pt' if (baseline_dir / 'latest.pt').exists() else None,
)
print('\nbaseline summary:', baseline_summary)

## 5. Final summary

Read the per-epoch validation records from each model's `train_log.jsonl` and print a side-by-side comparison. Detailed metrics (per-class accuracy, AUC, ROC, virtual-staining MSE/SSIM, qualitative figures) live in `notebooks/03_eval.ipynb`.

In [ ]:
import json

def load_val_records(log_path):
    if not log_path.exists():
        return []
    out = []
    with log_path.open() as f:
        for line in f:
            rec = json.loads(line)
            if rec.get('split') == 'val':
                out.append(rec)
    return out

cerberus_val = load_val_records(cerberus_dir / 'train_log.jsonl')
baseline_val = load_val_records(baseline_dir / 'train_log.jsonl')

print('Cerberus (brightfield-only multi-task)')
for r in cerberus_val:
    print(f"  epoch {r['epoch']}: acc_ct={r.get('acc_cell_type', float('nan')):.4f}  "
          f"acc_cond={r['acc_line_condition']:.4f}  "
          f"L_vs={r.get('L_virtual_staining', float('nan')):.4f}")

print('\nBaseline (6-channel single-task disease)')
for r in baseline_val:
    print(f"  epoch {r['epoch']}: acc_cond={r['acc_line_condition']:.4f}")

if cerberus_val and baseline_val:
    cerb_final = cerberus_val[-1]['acc_line_condition']
    base_final = baseline_val[-1]['acc_line_condition']
    print(f'\nDisease accuracy: Cerberus={cerb_final:.4f}, Baseline={base_final:.4f}, '
          f'gap={base_final - cerb_final:+.4f}')
    print('(Detailed AUC + significance tests in notebook 03.)')